In [1]:
import sys
from typing import List
import requests
import re
import numpy as np
from alfworld.agents.environment import get_environment
import alfworld.agents.modules.generic as generic


sys.argv = ['jupyter_notebook.py', 'configs/eval_config.yaml']
config = generic.load_config()
# env_type = config['env']['type']
# env = get_environment(env_type)(config, train_eval='train')
# env = env.init_env(batch_size=1)

In [2]:
from alfworld.agents.environment import get_environment
from alfworld.agents.environment.alfred_tw_env import TASK_TYPES
num_games = 50
eval_split = 'eval_out_of_distribution'
def load_games(game_files: List[str], num_games: int) -> List[str]:
    """Select games with uniform coverage across task types.

    If num_games <= 0, return all games sorted.
    Otherwise, take floor(num_games/num_types) per type,
    distribute the remainder round-robin by type alphabetical order.
    Within each type, take the first N (sorted) games.
    """
    files = sorted(game_files)
    if num_games <= 0 or num_games >= len(files):
        return files

    # Group by task type
    groups = {}
    for f in files:
        tt = detect_task_type(f)
        groups.setdefault(tt, []).append(f)

    types = sorted(groups.keys())
    per_type = num_games // len(types)
    remainder = num_games % len(types)

    selected = []
    for i, tt in enumerate(types):
        take = per_type + (1 if i < remainder else 0)
        selected.extend(groups[tt][:take])

    return sorted(selected)


def detect_task_type(gamefile_path: str) -> str:
    """Detect task type from the game file path."""
    for task_type in TASK_TYPES.values():
        if task_type in gamefile_path:
            return task_type
    return "unknown"

env_type = config["env"]["type"]
AlfredEnvClass = get_environment(env_type)
alfred_env = AlfredEnvClass(config, train_eval=eval_split)
alfred_env.game_files = load_games(alfred_env.game_files, num_games)
alfred_env.num_games = len(alfred_env.game_files)


Initializing AlfredTWEnv...


100%|██████████| 341/341 [00:00<00:00, 1279.56it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games


In [3]:
def call_llm(prompt):
    url = "https://tritonai-api.ucsd.edu/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + "sk-Z54pOkD5b_Y6b5jVORpDuw"
    }

    payload = {
    "model": "api-llama-4-scout",
    "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        result = response.json()
        return result['choices'][0]['message']['content'].strip()
    else:
        return f"API Error {response.status_code}: {response.text}"

## BaseLine 

In [4]:
# def alf_agent(task, obs, commands):
#     prompt = f"""You are a agent for a Text-based household simulation game
#     Task: {task}
#     Observation: {obs}
#     valid command: {commands}
#     You MUST return the text of the command you choose.
#     """
#     return call_llm(prompt)


### Evaluation

For baseline, or only LLM agent, the success rate is most likely 0.

In [5]:
# success_count = 0
# num_test_games = alfred_env.num_games
# MAX_STEPS = 50
# all_results = []

# for game_idx in range(num_test_games):
#     obs, info = env.reset()
#     task = obs[0].split("Your task is to: ")[1].strip()
#     valid_commands = list(info['admissible_commands'][0])

#     for _ in range(MAX_STEPS):
#         response = alf_agent(task, obs[0], valid_commands)
#         action = response
#         for cmd in valid_commands:
#             if cmd in response.lower():
#                 action = cmd
#                 break
#         obs, scores, dones, infos = env.step([action])
#         info = infos
#         if dones[0]:
#             if scores[0] > 0:
#                 success_count += 1
#                 print(f"success")
#             else:
#                 print(f"failure")
#             break
#         all_results.append(scores[0] > 0)

# final_sr = (success_count / num_test_games) * 100
# print(final_sr)

## LLM + Memory

In [6]:
def alf_agent_short_mem(task, obs, commands, history):
    recent_history = "\n".join([f"- {m}" for m in history]) if history else "None"
    
    prompt = f"""You are a agent for a Text-based household simulation game
    Task: {task}
    Observation: {obs}
    valid command: {commands}
    recent history: {recent_history}

    INSTRUCTION:
    1. Try to first find the object the task need.
    2. Check your history: Have you already tried this action? If it said "Nothing happens", do NOT repeat it.
    3. If you don't have the object, you must EXPLORE other locations.
    4. You MUST return the text of the command you choose.
    Output Format:
    Thought: <one sentence reasoning>
    Action: <command>
    """
    return call_llm(prompt)

## Memory + stateful knowledge base


In [7]:
def alf_agent_optimized(task, obs, commands, history, kb, current_subgoal):
    unvisited = [c for c in commands if "go to" in c and c.split("go to ")[1] not in kb['visited_locations']]
    
    inventory_status = f"HOLDING: {list(kb['inventory'])[0]}" if kb['inventory'] else "HANDS EMPTY"
    
    kb_summary = f"""
    [AGENT STATUS]
    - Current Hand: {inventory_status}
    - Searched (Dead Ends): {list(kb['visited_locations'])}
    - Unexplored Priority: {unvisited[:8]}
    """

    prompt = f"""You are a logical house agent in a simulated environment.
    
    TASK: {task}
    CURRENT OBSERVATION: {obs}
    VALID COMMANDS: {commands}

    {kb_summary}

    # RECENT HISTORY:
    {history if history else "Start of the game."}

    # MISSION SOP (Standard Operating Procedure):
    1. THE "ONE-HAND" LAW: You have ONLY ONE hand. If you are {inventory_status} and need to pick up a different item, you MUST 'put' or 'move' the current item down first. You CANNOT take a new item while holding one.
    2. MANUAL OPERATION: Putting an item in the microwave/sink and closing it is NOT enough. You MUST explicitly use 'heat/clean [obj] with [appliance]' as your next step. It NEVER happens automatically.
    3. NO GHOSTING: If you put an item in a container and close it, it is STILL THERE. Do not search the room again; trust your memory.
    4. SEARCH EFFICIENCY: If the target is NOT in 'CURRENT OBSERVATION', it is NOT there. Do not 'look' or 'examine' again. Immediately pick the first location from 'Unexplored Priority' and 'go to' it.

    ################################################################################
    # CRITICAL FINAL CHECK (Check before you decide!):
    # - Am I holding a useless item? (If yes, drop it now)
    # - Did I just close a microwave? (If yes, I MUST call the 'heat' command now)
    # - Is the target here? (If no, I MUST leave now)
    ################################################################################

    Current Sub-goal: {current_subgoal}

    Follow this format:
    Thought: <One sentence analyzing: My hand status + Observation vs Rules.>
    New Sub-goal: <Update based on rule compliance>
    Action: <Exactly one command from VALID COMMANDS>
    """
    return call_llm(prompt)

In [8]:
def update_kb(action, observation, kb):
    obs_text = observation[0] if isinstance(observation, (list, tuple)) else observation

    arrival_match = re.search(r"You arrive at ([\w\s\d\-]+)\.", obs_text)
    if arrival_match:
        loc = arrival_match.group(1).strip()
        kb["current_location"] = loc
        kb["visited_locations"].add(loc)
    
    if "you see" in obs_text.lower():
        loc = kb["current_location"]
        if loc != "unknown":
            items_part = obs_text.lower().split("you see")[-1].strip().rstrip(".")
            raw_items = re.split(r', and |and |, ', items_part)
            clean_items = [i.replace("a ", "").replace("an ", "").strip() for i in raw_items if i.strip()]
            old_items = kb["seen_items"].get(loc, [])
            kb["seen_items"][loc] = list(set(old_items + clean_items))

    if "you pick up the" in obs_text.lower():
        item_match = re.search(r"you pick up the ([\w\s\d]+) from", obs_text.lower())
        if item_match:
            item = item_match.group(1).strip()
            kb["inventory"].add(item)
            for l in kb["seen_items"]:
                kb["seen_items"][l] = [i for i in kb["seen_items"][l] if i != item]
    

    m = re.match(r"^(put|move)\s+(.+?)\s+to\s+(.+)$", action.lower())
    if m and ("You put" in obs_text or "You move" in obs_text):
        obj = m.group(2).strip()
        kb["inventory"] = {x for x in kb["inventory"] if x.lower() != obj}
        loc = kb["current_location"]
        if loc != "unknown":
            old_items = kb["seen_items"].get(loc, [])
            if obj not in old_items:
                kb["seen_items"][loc] = old_items + [obj]

    if kb["current_location"] == "unknown":
        print("Warning: Current location is unknown. Observation may be incomplete.")
    return kb

def extract_action_from_response(response):
    subgoal = "None"
    action_text = ""
    
    if "Current Sub-goal:" in response:
        parts = response.split("Current Sub-goal:")
        subgoal = parts[1].split("\n")[0].strip()
    
    if "Action:" in response:
        action_text = response.split("Action:")[-1].strip()
    else:
        lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
        action_text = lines[-1] if lines else ""
    
    return subgoal, action_text.rstrip('.,!?;:').strip().lower()

def validate_and_match_action(action_text, valid_commands):
    action_text = action_text.lower().strip()
    for cmd in valid_commands:
        if cmd.lower() == action_text:
            return cmd
        
    matches = [cmd for cmd in valid_commands if action_text in cmd.lower()]
    if len(matches) == 1:
        return matches[0]
        
    return None


In [9]:
import os
from datetime import datetime
import json
log_dir = "eval_logs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, f"eval_results_{datetime.now().strftime('%m%d_%H%M')}.json")

In [10]:
success_count = 0
num_test_games = alfred_env.num_games
MAX_STEPS = 50
MEMORY_SIZE = 6
all_results = []
tes = 25
all_game_logs = []
# Track success by game type
game_type_stats = {}

env = alfred_env.init_env(batch_size=1)
for game_idx in range(num_test_games):
    obs, info = env.reset()
    task = obs[0].split("Your task is to: ")[1].strip()
    short_memory = []
    current_subgoal = "None"
    kb = {
        "inventory": set(),       
        "visited_locations": set(),  
        "seen_items": {},      
        "current_location": "unknown",
    }
    
    # Extract game type from the game file
    game_file = alfred_env.game_files[game_idx]
    game_type = detect_task_type(game_file)
    
    # Initialize stats for this game type if not seen before
    if game_type not in game_type_stats:
        game_type_stats[game_type] = {"total": 0, "success": 0}
    game_type_stats[game_type]["total"] += 1
    
    current_game_record = {
        "game_idx": game_idx,
        "game_type": game_type,
        "task": task,
        "steps": [],
        "success": False
    }
    for step_num in range(MAX_STEPS):
        valid_commands = list(info['admissible_commands'][0])
        response = alf_agent_optimized(task, obs[0], valid_commands, short_memory, kb, current_subgoal)
        new_subgoal, action_text  = extract_action_from_response(response)
        action = validate_and_match_action(action_text, valid_commands)
        current_subgoal = new_subgoal

        if action == None:
            obs, scores, dones, infos = env.step(["look"])
            feedback = f"ERROR: '{action_text}' is NOT a valid command. (Environment Result: {obs[0]})"
            action = "look" 
        else:    
            obs, scores, dones, infos = env.step([action])
            feedback = obs[0]
            is_success = "nothing happens" not in feedback.lower()
            if not is_success:
                if action.startswith("take"):
                    feedback = f"FAILED: {feedback}. (Reason: Your hands are FULL. You MUST 'put' the item down first!)"
                else:
                    feedback = f"FAILED: {feedback}. (Reason: Check if the container is closed or item is missing.)"
            else:
                kb = update_kb(action, obs[0], kb)
        
        current_step_record = f"- Step {step_num}: {action} -> {feedback}"

        short_memory.append(current_step_record)
        if len(short_memory) > MEMORY_SIZE:
            short_memory.pop(0)

        info = infos
        step_data = {
            "step": step_num + 1,
            "llm_thought": response.split("Action:")[0].replace("Thought:", "").strip(),
            "chosen_action": action,
            "observation": obs,
            "score": scores[0],
        }
        current_game_record["steps"].append(step_data)
        if dones[0]:
            if scores[0] > 0:
                success_count += 1
                game_type_stats[game_type]["success"] += 1
                current_game_record["success"] = True
                print(f"success ({game_type})")
            else:
                print(f"failure ({game_type})")
            break
    all_game_logs.append(current_game_record)
    with open(log_file, 'w', encoding='utf-8') as f:
        json.dump(all_game_logs, f, indent=4, ensure_ascii=False)

final_sr = (success_count / num_test_games) * 100
print(f"\n{'='*50}")
print(f"Overall Success Rate: {final_sr:.2f}%")
print(f"{'='*50}")
print("\nSuccess Rate by Game Type:")
for game_type in sorted(game_type_stats.keys()):
    stats = game_type_stats[game_type]
    sr = (stats["success"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print(f"  {game_type:20s}: {stats['success']:3d}/{stats['total']:3d} = {sr:6.2f}%")
print(f"{'='*50}")

success (look_at_obj_in_light)
failure (look_at_obj_in_light)
failure (look_at_obj_in_light)
success (look_at_obj_in_light)
success (look_at_obj_in_light)
failure (look_at_obj_in_light)
success (look_at_obj_in_light)
success (look_at_obj_in_light)
failure (look_at_obj_in_light)
failure (pick_and_place_simple)
failure (pick_and_place_simple)
success (pick_and_place_simple)
success (pick_and_place_simple)
failure (pick_and_place_simple)
success (pick_and_place_simple)
success (pick_and_place_simple)
success (pick_and_place_simple)
success (pick_and_place_simple)
success (pick_clean_then_place_in_recep)
failure (pick_clean_then_place_in_recep)
success (pick_clean_then_place_in_recep)
failure (pick_clean_then_place_in_recep)
success (pick_clean_then_place_in_recep)
failure (pick_clean_then_place_in_recep)
success (pick_clean_then_place_in_recep)
failure (pick_clean_then_place_in_recep)
failure (pick_cool_then_place_in_recep)
failure (pick_cool_then_place_in_recep)
failure (pick_cool_then_p